# Percobaan 02: Ekstraksi Fitur Tekstur
Notebook ini berisi proses ekstraksi fitur tekstur dari citra hasil preprocessing menggunakan metode Gray Level Co-occurrence Matrix atau GLCM. Setiap citra dianalisis berdasarkan hubungan pasangan piksel pada beberapa arah sudut. Fitur yang dihasilkan digunakan untuk merepresentasikan tekstur citra dalam bentuk data numerik. Data numerik tersebut kemudian disimpan ke file CSV agar dapat digunakan pada tahap klasifikasi.


## Import Library
Bagian ini mengimpor library yang dibutuhkan untuk membaca citra, mengelola file, menghitung fitur tekstur, dan menyimpan data hasil ekstraksi. Library `os` dan `pathlib` digunakan untuk mengakses folder hasil preprocessing dan mengatur lokasi output. Library `cv2` digunakan untuk membaca citra dalam format grayscale. Library `numpy` digunakan untuk operasi numerik pada matriks citra dan hasil GLCM. Library `pandas` digunakan untuk menyusun hasil ekstraksi dalam bentuk tabel, sedangkan `skimage` dan `scipy` digunakan untuk menghitung matriks GLCM serta nilai entropi.


In [1]:
import os
import cv2 as cv
import numpy as np
import pandas as pd
from pathlib import Path
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy

## Konfigurasi Folder
Bagian ini menentukan folder input yang berisi hasil preprocessing dan folder output untuk menyimpan hasil ekstraksi fitur. Folder `hasil_ekstraksi` dibuat otomatis jika belum tersedia agar proses penyimpanan CSV tidak gagal. Dictionary `PREPO_CONFIG` menghubungkan nama folder preprocessing dengan nama file CSV yang akan dihasilkan. Daftar `VALID_EXTENSIONS` digunakan untuk membatasi file yang diproses hanya pada format citra. Program juga menampilkan status keberadaan setiap folder preprocessing agar kesalahan path dapat diketahui sejak awal.


In [2]:
from pathlib import Path

# Folder input: hasil preprocessing
PREPROCESSING_DIR = Path("../preprocessing_output")

# Folder output: hasil ekstraksi fitur CSV
OUTPUT_DIR = Path("../hasil_ekstraksi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Folder hasil preprocessing yang akan diekstraksi fiturnya
PREPO_CONFIG = {
    "prepo1_resize+grayscale": "hasil_ekstraksi_prepo1.csv",
    "prepo2_resize+grayscale+median": "hasil_ekstraksi_prepo2.csv",
    "prepo3_resize+grayscale+median+equ": "hasil_ekstraksi_prepo3.csv",
    "prepo4_resize+grayscale+median+sobel": "hasil_ekstraksi_prepo4.csv",
    "prepo5_resize+grayscale+median+sobel+thresholding": "hasil_ekstraksi_prepo5.csv"
}

VALID_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp"]

print("Folder preprocessing:", PREPROCESSING_DIR.resolve())
print("Folder hasil ekstraksi:", OUTPUT_DIR.resolve())

for prepo_name in PREPO_CONFIG:
    prepo_path = PREPROCESSING_DIR / prepo_name
    print(prepo_name, "->", "ADA" if prepo_path.exists() else "TIDAK ADA")

Folder preprocessing: D:\ASUS\Documents\SEMS 4\Praktikum PCD\Projek\Project-PCD-Kelompok-16\preprocessing_output
Folder hasil ekstraksi: D:\ASUS\Documents\SEMS 4\Praktikum PCD\Projek\Project-PCD-Kelompok-16\hasil_ekstraksi
prepo1_resize+grayscale -> ADA
prepo2_resize+grayscale+median -> ADA
prepo3_resize+grayscale+median+equ -> ADA
prepo4_resize+grayscale+median+sobel -> ADA
prepo5_resize+grayscale+median+sobel+thresholding -> ADA


## Fungsi Perhitungan GLCM
Fungsi `glcm()` membentuk matriks Gray Level Co-occurrence Matrix berdasarkan sudut 0, 45, 90, atau 135 derajat. Parameter `distances=[1]` berarti hubungan piksel dihitung berdasarkan jarak satu piksel dari piksel acuan. Matriks dibuat dengan 256 level keabuan karena citra grayscale memiliki rentang nilai 0 sampai 255. Fungsi `safe_value()` memastikan nilai hasil perhitungan tidak berupa NaN atau infinity sehingga aman disimpan ke dataset fitur. Fungsi lain seperti `contrast`, `homogenity`, `energy`, `correlation`, `dissimilarity`, `ASM`, dan `entropyGlcm` digunakan untuk mengambil ukuran tekstur dari matriks GLCM.


In [3]:
def glcm(image, derajat):
    if derajat == 0:
        angles = [0]
    elif derajat == 45:
        angles = [np.pi / 4]
    elif derajat == 90:
        angles = [np.pi / 2]
    elif derajat == 135:
        angles = [3 * np.pi / 4]
    else:
        raise ValueError("Derajat harus salah satu dari: 0, 45, 90, 135.")

    matriks_glcm = graycomatrix(
        image,
        distances=[1],
        angles=angles,
        levels=256,
        symmetric=True,
        normed=True
    )

    return matriks_glcm


def safe_value(value):
    value = float(value)
    if np.isnan(value) or np.isinf(value):
        return 0.0
    return value


def correlation(matriks):
    return safe_value(graycoprops(matriks, 'correlation')[0, 0])


def dissimilarity(matriks):
    return safe_value(graycoprops(matriks, 'dissimilarity')[0, 0])


def homogenity(matriks):
    return safe_value(graycoprops(matriks, 'homogeneity')[0, 0])


def contrast(matriks):
    return safe_value(graycoprops(matriks, 'contrast')[0, 0])


def ASM(matriks):
    return safe_value(graycoprops(matriks, 'ASM')[0, 0])


def energy(matriks):
    return safe_value(graycoprops(matriks, 'energy')[0, 0])


def entropyGlcm(matriks):
    return safe_value(entropy(matriks.ravel()))

## Ekstraksi Fitur per Citra
Fungsi `extract_glcm_features()` digunakan untuk menghitung seluruh fitur tekstur pada satu citra. Proses dilakukan pada empat sudut, yaitu 0, 45, 90, dan 135 derajat, agar pola tekstur dari berbagai arah dapat terwakili. Pada setiap sudut, fungsi membentuk matriks GLCM terlebih dahulu lalu menghitung contrast, homogeneity, dissimilarity, entropy, ASM, energy, dan correlation. Nama kolom fitur diberi akhiran sudut, misalnya `Contrast0` atau `Energy135`, supaya setiap nilai mudah dibedakan. Hasil akhirnya berupa dictionary yang berisi kumpulan fitur numerik untuk satu citra.


In [4]:
def extract_glcm_features(image):
    angles = [0, 45, 90, 135]
    features = {}

    for angle in angles:
        matriks = glcm(image, angle)

        features[f"Contrast{angle}"] = contrast(matriks)
        features[f"Homogeneity{angle}"] = homogenity(matriks)
        features[f"Dissimilarity{angle}"] = dissimilarity(matriks)
        features[f"Entropy{angle}"] = entropyGlcm(matriks)
        features[f"ASM{angle}"] = ASM(matriks)
        features[f"Energy{angle}"] = energy(matriks)
        features[f"Correlation{angle}"] = correlation(matriks)

    return features

## Ekstraksi Fitur per Folder
Fungsi `extract_features_from_preprocessing_folder()` membaca semua citra dari satu folder metode preprocessing. Program menelusuri subfolder kelas, membaca file gambar yang valid dalam mode grayscale, lalu mengubahnya menjadi tipe `uint8`. Setiap citra diproses menggunakan `extract_glcm_features()` untuk mendapatkan fitur tekstur. Informasi nama file, label kelas, dan nama preprocessing ditambahkan agar setiap baris data tetap memiliki identitas asal citra. Semua baris fitur disusun menjadi DataFrame dan disimpan sebagai file CSV pada folder `hasil_ekstraksi`. Jika folder input tidak ditemukan atau citra gagal dibaca, program menampilkan pesan agar proses dapat diperiksa.


In [5]:
def extract_features_from_preprocessing_folder(prepo_name, output_csv_name):
    prepo_path = PREPROCESSING_DIR / prepo_name

    if not prepo_path.exists():
        print(f"Folder tidak ditemukan: {prepo_path}")
        return None

    rows = []

    for class_folder in sorted(prepo_path.iterdir()):
        if not class_folder.is_dir():
            continue

        label = class_folder.name

        image_files = [
            file for file in sorted(class_folder.iterdir())
            if file.suffix.lower() in VALID_EXTENSIONS
        ]

        print(f"{prepo_name} | {label}: {len(image_files)} gambar")

        for image_path in image_files:
            image = cv.imread(str(image_path), cv.IMREAD_GRAYSCALE)

            if image is None:
                print(f"Gagal membaca gambar: {image_path}")
                continue

            image = image.astype(np.uint8)

            feature_data = extract_glcm_features(image)

            row = {
                "Filename": image_path.name,
                "Label": label,
                "Preprocessing": prepo_name
            }

            row.update(feature_data)
            rows.append(row)

    df = pd.DataFrame(rows)

    output_csv_path = OUTPUT_DIR / output_csv_name
    df.to_csv(output_csv_path, index=False)

    print(f"CSV berhasil disimpan: {output_csv_path}")
    print(f"Jumlah data: {len(df)}")

    return df

## Menjalankan Seluruh Ekstraksi
Bagian ini menjalankan proses ekstraksi fitur untuk semua metode preprocessing yang terdaftar dalam `PREPO_CONFIG`. Setiap metode diproses satu per satu dengan memanggil fungsi `extract_features_from_preprocessing_folder()`. Hasil DataFrame yang berhasil dibuat disimpan ke dalam dictionary `hasil_ekstraksi` dengan nama preprocessing sebagai kunci. Tampilan pemisah pada output digunakan agar proses tiap metode lebih mudah dibaca. Setelah seluruh metode selesai, program menampilkan pesan bahwa proses ekstraksi fitur telah selesai dilakukan.


In [6]:
hasil_ekstraksi = {}

for prepo_name, output_csv_name in PREPO_CONFIG.items():
    print("\n" + "=" * 80)
    print(f"Ekstraksi fitur: {prepo_name}")
    print("=" * 80)

    df = extract_features_from_preprocessing_folder(prepo_name, output_csv_name)

    if df is not None:
        hasil_ekstraksi[prepo_name] = df

print("\nSemua ekstraksi fitur selesai.")


Ekstraksi fitur: prepo1_resize+grayscale
prepo1_resize+grayscale | catterpillar: 70 gambar


prepo1_resize+grayscale | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo1.csv
Jumlah data: 140

Ekstraksi fitur: prepo2_resize+grayscale+median
prepo2_resize+grayscale+median | catterpillar: 70 gambar
prepo2_resize+grayscale+median | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo2.csv
Jumlah data: 140

Ekstraksi fitur: prepo3_resize+grayscale+median+equ
prepo3_resize+grayscale+median+equ | catterpillar: 70 gambar
prepo3_resize+grayscale+median+equ | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo3.csv
Jumlah data: 140

Ekstraksi fitur: prepo4_resize+grayscale+median+sobel
prepo4_resize+grayscale+median+sobel | catterpillar: 70 gambar
prepo4_resize+grayscale+median+sobel | snail: 70 gambar
CSV berhasil disimpan: ..\hasil_ekstraksi\hasil_ekstraksi_prepo4.csv
Jumlah data: 140

Ekstraksi fitur: prepo5_resize+grayscale+median+sobel+thresholding
prepo5_resize+grayscale+median+sobel+thresholdi